# Historian Events Loader

This notebook processes the historian and events data for a given alarm tag.

**Workflow:**
1. Load the alarm tag's related-tag information from the Excel workbook (`UC3_Alarm_Tags_Presentation_30MAY2026.xlsx`).
2. Extract DCS tag names and their associated asset IDs from the sheet.
3. Map those short asset IDs (e.g. `1E`, `1F`) to UUIDs using the `ADNOC-B.json` config file.
4. Discover which UUID folders exist under `Historian_Events/Historian/` and `Historian_Events/events/`.
5. Load the historian time-series parquet files for each related tag.
6. Load the event parquet files (_E, _EA, _EAL) for each asset.

**Target tag for this run:** `03TIC_1023` (sheet: `03TIC_1023 PVLO_PVHI`)

## 0. Configuration

In [29]:
# ── Paths ──────────────────────────────────────────────────────────────────────
import os

REPO_ROOT          = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), '.'))
DATA_DIR           = os.path.join(REPO_ROOT, 'DATA')
HISTORIAN_ROOT     = os.path.join(DATA_DIR, 'Historian_Events')
HISTORIAN_DIR      = os.path.join(HISTORIAN_ROOT, 'Historian')   # tag time-series
EVENTS_DIR         = os.path.join(HISTORIAN_ROOT, 'events')       # alarm/event tables
EXCEL_FILE         = os.path.join(DATA_DIR, 'UC3_Alarm_Tags_Presentation_30MAY2026.xlsx')
ADNOC_JSON         = os.path.join(DATA_DIR, 'config', 'config', 'EMDB', 'ADNOC-B.json')

# ── Target alarm tag ──────────────────────────────────────────────────────────
TARGET_TAG         = '03LIC_1619'
EXCEL_HEADER_ROW   = 6      # 0-indexed row that contains column headers

# Auto-detect the sheet whose name contains TARGET_TAG
import openpyxl
_wb = openpyxl.load_workbook(EXCEL_FILE, read_only=True, data_only=True)
_matches = [s for s in _wb.sheetnames if TARGET_TAG in s]
_wb.close()
if not _matches:
    raise ValueError(f"No sheet found containing '{TARGET_TAG}' in {EXCEL_FILE}")
SHEET_NAME = _matches[0]

print(f"REPO_ROOT      : {REPO_ROOT}")
print(f"EXCEL_FILE     : {EXCEL_FILE}")
print(f"ADNOC_JSON     : {ADNOC_JSON}")
print(f"HISTORIAN_DIR  : {HISTORIAN_DIR}")
print(f"EVENTS_DIR     : {EVENTS_DIR}")
print(f"TARGET_TAG     : {TARGET_TAG}")
print(f"SHEET_NAME     : {SHEET_NAME}  (auto-detected)")

REPO_ROOT      : /home/h604827/ControlActions
EXCEL_FILE     : /home/h604827/ControlActions/DATA/UC3_Alarm_Tags_Presentation_30MAY2026.xlsx
ADNOC_JSON     : /home/h604827/ControlActions/DATA/config/config/EMDB/ADNOC-B.json
HISTORIAN_DIR  : /home/h604827/ControlActions/DATA/Historian_Events/Historian
EVENTS_DIR     : /home/h604827/ControlActions/DATA/Historian_Events/events
TARGET_TAG     : 03LIC_1619
SHEET_NAME     : 03LIC_1619 PVLO_PVHI  (auto-detected)


## 1. Imports

In [30]:
import json
import pandas as pd

print(f"pandas  : {pd.__version__}")

pandas  : 2.3.3


## 2. Load Related-Tag Information from Excel

Each sheet in the Excel workbook describes the related tags for one alarm.  
We extract **DCS Tag Name**, **DESCRIPTION**, **PARAMETER**, **IS DATA AVAILABLE?** and **ASSET ID**.

In [31]:
print(f"Reading sheet '{SHEET_NAME}' from:\n  {EXCEL_FILE}\n")

df_sheet = pd.read_excel(EXCEL_FILE, sheet_name=SHEET_NAME, header=EXCEL_HEADER_ROW)

print(f"Raw sheet shape  : {df_sheet.shape}")
print(f"Columns ({len(df_sheet.columns)}):")
for c in df_sheet.columns:
    print(f"  {c}")

Reading sheet '03LIC_1619 PVLO_PVHI' from:
  /home/h604827/ControlActions/DATA/UC3_Alarm_Tags_Presentation_30MAY2026.xlsx

Raw sheet shape  : (12, 29)
Columns (29):
  Remarks
  Potential Input parameter
  Description
  Prabable Cause (PVHI)
  Probable Causes (PVLO)
  Plant
  P&ID
  DCS (Yes/No)
  SME Comment
  Unnamed: 9
  Manish Comments
  VS Comments
  DCS Tag Name
  DESCRIPTION
  PARAMETER
  IS DATA AVAILABLE?
  SP-HIGH LIMIT
  SP-LOW LIMIT
  EXTENDED PV HIGH LIMIT
  PV-HIGH LIMIT
  EXTENDED PV LOW LIMIT
  PV-LOW LIMIT
  OP-HIGH LIMIT
  OP-LOW LIMIT
  PHD_TAG AVILABLE
  FILE NAME
  ASSET ID
  AVAILABLE IN APC
  SAFETY MANAGER TAGS


In [32]:
# Keep only rows that have a DCS Tag Name
KEEP_COLS = ['Potential Input parameter', 'DCS Tag Name', 'DESCRIPTION', 'PARAMETER',
             'IS DATA AVAILABLE?', 'ASSET ID']

df_tags = (
    df_sheet[KEEP_COLS]
    .dropna(subset=['DCS Tag Name'])
    .copy()
)

# Normalise strings
df_tags['DCS Tag Name'] = df_tags['DCS Tag Name'].str.strip()
df_tags['ASSET ID']     = df_tags['ASSET ID'].astype(str).str.strip()

# Reset index for clean display
df_tags = df_tags.reset_index(drop=True)

print(f"Related tags found: {len(df_tags)}")
print()
display(df_tags)

Related tags found: 12



,Potential Input parameter,DCS Tag Name,DESCRIPTION,PARAMETER,IS DATA AVAILABLE?,ASSET ID
0,03LIC_1619,03LIC_1619,3C116 GLYCOL LEVEL,PV,YES,1O
1,03LICA_1608,03LIC_1608,3C111 GLYCOL LEVEL,PV,YES,1O
2,03PIC_1620,03PIC_1620,3C116 PRES CNTRL VALVE,PV,YES,1O
3,03FICA_1668,03FIC_1668,3E113A/B GLYCOL FLW CTRL,PV,YES,1O
4,03TIC_1671,03TIC_1671,3E112 O/L TEMP CNTRL,PV,YES,1O
5,03LICA_1603,03LIC_1603,3C111 H/C LEVEL CNTRL,OP,YES,1O
6,03LICA_1618,03LIC_1618,3C116 H/C LEVEL,OP,YES,1O
7,03G116,03GM_0116_I,TR-1 GLYCOL PMP,PV,YES,1O
8,03G116A,03GM_0116A_I,TR-1 GLYCOL PMP-A,PV,YES,1O
9,02FI_1000,02FI_1000,TRAIN 1 FEED,PV,YES,1A


## 3. Load ADNOC-B.json and Build Asset-ID → UUID Mapping

The JSON config contains one entry per asset node in the ADNOC Buhasa plant hierarchy.  
Leaf nodes whose `Name` matches the short asset ID in the sheet provide the UUID we need.

In [33]:
print(f"Reading asset config:\n  {ADNOC_JSON}\n")

with open(ADNOC_JSON, 'r') as f:
    asset_nodes = json.load(f)

print(f"Total nodes in ADNOC-B.json : {len(asset_nodes)}")

# Build a flat map:  short-name (e.g. '1E') → UUID
# We only care about leaf nodes under TRAIN_1 (adjust if other areas are needed)
asset_id_to_uuid = {}
for node in asset_nodes:
    asset_id_to_uuid[node['Name']] = {
        'uuid'      : node['Id'],
        'hierarchy' : node['Hierarchy'],
        'leaf'      : node['LeafNode'],
    }

print(f"Unique asset names mapped    : {len(asset_id_to_uuid)}")
print()

# Show all leaf nodes for easy inspection
leaf_map = pd.DataFrame(
    [{'Asset Name': k, 'UUID': v['uuid'], 'Hierarchy': v['hierarchy'], 'IsLeaf': v['leaf']}
     for k, v in asset_id_to_uuid.items()]
).sort_values('Asset Name').reset_index(drop=True)

print("Full asset map (all nodes):")
display(leaf_map)

Reading asset config:
  /home/h604827/ControlActions/DATA/config/config/EMDB/ADNOC-B.json

Total nodes in ADNOC-B.json : 74
Unique asset names mapped    : 74

Full asset map (all nodes):


,Asset Name,UUID,Hierarchy,IsLeaf
0,1A,8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3,ADNOC\BUHASA\ADNOC-B\TRAIN_1\1A,True
1,1B,f0c70dd0-139e-4102-b75d-77bccee3ca83,ADNOC\BUHASA\ADNOC-B\TRAIN_1\1B,True
2,1C,c71817e8-5df3-4ad6-89fb-0b71ec7d9f96,ADNOC\BUHASA\ADNOC-B\TRAIN_1\1C,True
3,1D,9cac06f4-131b-4af0-aea0-17b223973d12,ADNOC\BUHASA\ADNOC-B\TRAIN_1\1D,True
4,1E,072404dc-ae93-4239-9f6b-f4624391c391,ADNOC\BUHASA\ADNOC-B\TRAIN_1\1E,True
...,...,...,...,...
69,FIRE & GAS,de29a368-6e72-48a4-bbd2-84c32d2ac510,ADNOC\BUHASA\ADNOC-B\FIRE & GAS,False
70,LEAN_GAS,a663c56b-aafd-4a6a-837f-c8677655e9ca,ADNOC\BUHASA\ADNOC-B\LEAN_GAS,False
71,TRAIN_1,cdfe1dc7-d3db-4efe-a9b6-c5dec2902ccb,ADNOC\BUHASA\ADNOC-B\TRAIN_1,False
72,TRAIN_2,cc6268d1-0fe2-4fc2-8d84-79f6080b172d,ADNOC\BUHASA\ADNOC-B\TRAIN_2,False


## 4. Map Sheet Asset IDs → UUIDs

For every unique asset ID referenced in the sheet, look up the corresponding UUID.

In [34]:
unique_assets = sorted(df_tags['ASSET ID'].dropna().unique())
print(f"Unique ASSET IDs in sheet : {unique_assets}")
print()

mapping_rows = []
for asset_name in unique_assets:
    if asset_name in asset_id_to_uuid:
        info = asset_id_to_uuid[asset_name]
        mapping_rows.append({
            'Asset Name' : asset_name,
            'UUID'       : info['uuid'],
            'Hierarchy'  : info['hierarchy'],
        })
        print(f"  {asset_name:6s}  →  {info['uuid']}  ({info['hierarchy']})")
    else:
        mapping_rows.append({'Asset Name': asset_name, 'UUID': None, 'Hierarchy': None})
        print(f"  {asset_name:6s}  →  [NOT FOUND in ADNOC-B.json]")

df_asset_map = pd.DataFrame(mapping_rows)
print()
print(f"Resolved: {df_asset_map['UUID'].notna().sum()} / {len(df_asset_map)} assets")

Unique ASSET IDs in sheet : ['1A', '1O']

  1A      →  8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3  (ADNOC\BUHASA\ADNOC-B\TRAIN_1\1A)
  1O      →  df0dd88a-ed6d-412b-9d36-6c35829939cd  (ADNOC\BUHASA\ADNOC-B\TRAIN_1\1O)

Resolved: 2 / 2 assets


In [35]:
# Attach UUID back to each row in df_tags for convenience
df_tags = df_tags.merge(
    df_asset_map.rename(columns={'Asset Name': 'ASSET ID'})[['ASSET ID', 'UUID']],
    on='ASSET ID', how='left'
)

print("df_tags enriched with UUID column:")
display(df_tags[['DCS Tag Name', 'DESCRIPTION', 'ASSET ID', 'UUID']])

df_tags enriched with UUID column:


,DCS Tag Name,DESCRIPTION,ASSET ID,UUID
0,03LIC_1619,3C116 GLYCOL LEVEL,1O,df0dd88a-ed6d-412b-9d36-6c35829939cd
1,03LIC_1608,3C111 GLYCOL LEVEL,1O,df0dd88a-ed6d-412b-9d36-6c35829939cd
2,03PIC_1620,3C116 PRES CNTRL VALVE,1O,df0dd88a-ed6d-412b-9d36-6c35829939cd
3,03FIC_1668,3E113A/B GLYCOL FLW CTRL,1O,df0dd88a-ed6d-412b-9d36-6c35829939cd
4,03TIC_1671,3E112 O/L TEMP CNTRL,1O,df0dd88a-ed6d-412b-9d36-6c35829939cd
5,03LIC_1603,3C111 H/C LEVEL CNTRL,1O,df0dd88a-ed6d-412b-9d36-6c35829939cd
6,03LIC_1618,3C116 H/C LEVEL,1O,df0dd88a-ed6d-412b-9d36-6c35829939cd
7,03GM_0116_I,TR-1 GLYCOL PMP,1O,df0dd88a-ed6d-412b-9d36-6c35829939cd
8,03GM_0116A_I,TR-1 GLYCOL PMP-A,1O,df0dd88a-ed6d-412b-9d36-6c35829939cd
9,02FI_1000,TRAIN 1 FEED,1A,8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3


## 5. Discover Available Folders in Historian_Events

The `Historian/` sub-folder holds **tag-level time-series** parquet files (one file per tag per asset).  
The `events/` sub-folder holds **alarm & event** parquet files for each asset.

We check which asset UUID folders actually exist on disk.

In [36]:
historian_folders = set(os.listdir(HISTORIAN_DIR))
events_folders    = set(os.listdir(EVENTS_DIR))

print(f"Folders found in Historian/ : {len(historian_folders)}")
print(f"Folders found in events/    : {len(events_folders)}")
print()

availability = []
for _, row in df_asset_map.iterrows():
    uuid = row['UUID']
    in_hist   = uuid in historian_folders if uuid else False
    in_events = uuid in events_folders    if uuid else False
    availability.append({
        'Asset Name'      : row['Asset Name'],
        'UUID'            : uuid,
        'In Historian/'   : in_hist,
        'In events/'      : in_events,
    })
    status = []
    if in_hist:   status.append('Historian/')
    if in_events: status.append('events/')
    status_str = ', '.join(status) if status else 'MISSING in both'
    print(f"  {row['Asset Name']:6s}  {uuid}  →  {status_str}")

df_availability = pd.DataFrame(availability)
print()
display(df_availability)

Folders found in Historian/ : 13
Folders found in events/    : 70

  1A      8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3  →  Historian/, events/
  1O      df0dd88a-ed6d-412b-9d36-6c35829939cd  →  Historian/, events/



,Asset Name,UUID,In Historian/,In events/
0,1A,8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3,True,True
1,1O,df0dd88a-ed6d-412b-9d36-6c35829939cd,True,True


## 6. Load Historian Time-Series Data

For each asset that has a folder in `Historian/`, list the available tag files and load  
the ones corresponding to the tags referenced in the sheet.

Each parquet file in `Historian/<UUID>/` is named `<TAG>.<PARAMETER>` (e.g. `03TIC_1023.PV`).  
Columns: `startdatetime`, `enddatetime`, `timestamp`, `value`, `S`.

In [37]:
historian_data = {}   # key: 'TAG.PARAM'  →  value: DataFrame

for _, asset_row in df_asset_map.iterrows():
    uuid       = asset_row['UUID']
    asset_name = asset_row['Asset Name']

    if not uuid or uuid not in historian_folders:
        print(f"[SKIP] {asset_name} — no Historian folder")
        continue

    asset_hist_dir = os.path.join(HISTORIAN_DIR, uuid)
    available_files = os.listdir(asset_hist_dir)

    # Tags in the sheet that belong to this asset
    asset_tags = df_tags.loc[df_tags['UUID'] == uuid, 'DCS Tag Name'].tolist()

    print(f"\n── Asset {asset_name} ({uuid}) ──")
    print(f"   Files in Historian folder : {len(available_files)}")
    print(f"   Tags in sheet for asset   : {asset_tags}")

    for tag in asset_tags:
        param = df_tags.loc[df_tags['DCS Tag Name'] == tag, 'PARAMETER'].values
        param = param[0] if len(param) > 0 else 'PV'
        filename = f"{tag}.{param}"

        filepath = os.path.join(asset_hist_dir, filename)
        if os.path.exists(filepath):
            df_hist = pd.read_parquet(filepath)
            df_hist['timestamp'] = pd.to_datetime(df_hist['timestamp'])
            historian_data[filename] = df_hist
            print(f"   ✓ Loaded  {filename:30s}  →  shape {df_hist.shape}  "
                  f"| range: {df_hist['timestamp'].min()} … {df_hist['timestamp'].max()}")
        else:
            # Try scanning available files for a match
            matches = [f for f in available_files if f.startswith(tag + '.')]
            if matches:
                for m in matches:
                    df_hist = pd.read_parquet(os.path.join(asset_hist_dir, m))
                    df_hist['timestamp'] = pd.to_datetime(df_hist['timestamp'])
                    historian_data[m] = df_hist
                    print(f"   ✓ Loaded  {m:30s}  →  shape {df_hist.shape}  "
                          f"| range: {df_hist['timestamp'].min()} … {df_hist['timestamp'].max()}")
            else:
                print(f"   ✗ Not found: {filename}")

print(f"\n{'='*60}")
print(f"Total historian series loaded: {len(historian_data)}")


── Asset 1A (8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3) ──
   Files in Historian folder : 11
   Tags in sheet for asset   : ['02FI_1000']
   ✓ Loaded  02FI_1000.PV                    →  shape (2130628, 5)  | range: 2022-01-04 01:00:00 … 2026-01-10 01:00:00

── Asset 1O (df0dd88a-ed6d-412b-9d36-6c35829939cd) ──
   Files in Historian folder : 30
   Tags in sheet for asset   : ['03LIC_1619', '03LIC_1608', '03PIC_1620', '03FIC_1668', '03TIC_1671', '03LIC_1603', '03LIC_1618', '03GM_0116_I', '03GM_0116A_I', '03PDI_1611', '03TIC_1635']
   ✓ Loaded  03LIC_1619.PV                   →  shape (2134309, 5)  | range: 2022-01-04 01:00:00 … 2026-01-10 01:00:00
   ✓ Loaded  03LIC_1608.PV                   →  shape (2134309, 5)  | range: 2022-01-04 01:00:00 … 2026-01-10 01:00:00
   ✓ Loaded  03PIC_1620.PV                   →  shape (2134280, 5)  | range: 2022-01-04 01:00:00 … 2026-01-10 01:00:00
   ✓ Loaded  03FIC_1668.PV                   →  shape (2112127, 5)  | range: 2022-01-04 01:00:00 … 2026-01-10 01

### 6a. Historian Data — Preview

In [38]:
for key, df_h in historian_data.items():
    print(f"\n── {key} ──")
    print(f"   Shape   : {df_h.shape}")
    print(f"   Columns : {df_h.columns.tolist()}")
    print(f"   Null %  : {(df_h['value'].isna().mean()*100):.2f}%  missing values")
    display(df_h.head(3))


── 02FI_1000.PV ──
   Shape   : (2130628, 5)
   Columns : ['startdatetime', 'enddatetime', 'timestamp', 'value', 'S']
   Null %  : 98.36%  missing values


,startdatetime,enddatetime,timestamp,value,S
0,2022-01-04 00:00:00,2022-01-04 01:00:00,2022-01-04 01:00:00,6.623204,NaN
1,2022-01-04 01:00:00,2022-01-04 02:00:00,2022-01-04 02:00:00,6.683493,NaN
2,2022-01-04 02:00:00,2022-01-04 03:00:00,2022-01-04 03:00:00,6.603747,NaN



── 03LIC_1619.PV ──
   Shape   : (2134309, 5)
   Columns : ['startdatetime', 'enddatetime', 'timestamp', 'value', 'S']
   Null %  : 98.36%  missing values


,startdatetime,enddatetime,timestamp,value,S
0,2022-01-04 00:00:00,2022-01-04 01:00:00,2022-01-04 01:00:00,34.585071,NaN
1,2022-01-04 01:00:00,2022-01-04 02:00:00,2022-01-04 02:00:00,35.165872,NaN
2,2022-01-04 02:00:00,2022-01-04 03:00:00,2022-01-04 03:00:00,34.911793,NaN



── 03LIC_1608.PV ──
   Shape   : (2134309, 5)
   Columns : ['startdatetime', 'enddatetime', 'timestamp', 'value', 'S']
   Null %  : 98.36%  missing values


...

,startdatetime,enddatetime,timestamp,value,S
0,2022-01-04 00:00:00,2022-01-04 01:00:00,2022-01-04 01:00:00,-0.027258,NaN
1,2022-01-04 01:00:00,2022-01-04 02:00:00,2022-01-04 02:00:00,-0.019637,NaN
2,2022-01-04 02:00:00,2022-01-04 03:00:00,2022-01-04 03:00:00,-0.033747,NaN



── 03PDI_1611.PV ──
   Shape   : (2134299, 5)
   Columns : ['startdatetime', 'enddatetime', 'timestamp', 'value', 'S']
   Null %  : 98.36%  missing values


,startdatetime,enddatetime,timestamp,value,S
0,2022-01-04 00:00:00,2022-01-04 01:00:00,2022-01-04 01:00:00,59.752084,NaN
1,2022-01-04 01:00:00,2022-01-04 02:00:00,2022-01-04 02:00:00,60.586028,NaN
2,2022-01-04 02:00:00,2022-01-04 03:00:00,2022-01-04 03:00:00,59.960373,NaN



── 03TIC_1635.PV ──
   Shape   : (2134126, 5)
   Columns : ['startdatetime', 'enddatetime', 'timestamp', 'value', 'S']
   Null %  : 98.36%  missing values


,startdatetime,enddatetime,timestamp,value,S
0,2022-01-04 00:00:00,2022-01-04 01:00:00,2022-01-04 01:00:00,186.654339,NaN
1,2022-01-04 01:00:00,2022-01-04 02:00:00,2022-01-04 02:00:00,186.706065,NaN
2,2022-01-04 02:00:00,2022-01-04 03:00:00,2022-01-04 03:00:00,185.461431,NaN


## 7. Load Events Data

For each asset UUID folder in `events/`, there are up to three types of parquet files:

| Suffix | Content |
|--------|---------|
| `_E`   | Process/operator change events |
| `_EA`  | Alarm events (no last/next linkage) |
| `_EAL` | Alarm events **with** linked previous/next alarm (includes duration) |

We load all three types for every relevant asset.

In [39]:
events_data = {}   # key: '<asset_name>_<type>'  →  value: DataFrame

for _, asset_row in df_asset_map.iterrows():
    uuid       = asset_row['UUID']
    asset_name = asset_row['Asset Name']

    if not uuid or uuid not in events_folders:
        print(f"[SKIP] {asset_name} — no events folder")
        continue

    asset_events_dir = os.path.join(EVENTS_DIR, uuid)
    available_files  = os.listdir(asset_events_dir)

    print(f"\n── Asset {asset_name} ({uuid}) ──")
    print(f"   Files in events folder : {available_files}")

    for suffix, label in [('_E.parquet', 'ChangeEvents'), ('_EA.parquet', 'AlarmEvents'), ('_EAL.parquet', 'AlarmEventsLinked')]:
        match = [f for f in available_files if f.endswith(suffix)]
        if match:
            fp = os.path.join(asset_events_dir, match[0])
            df_ev = pd.read_parquet(fp)
            # Parse timestamp column where available — use format='mixed' to handle
            # rows where milliseconds are present/absent in the same column
            for ts_col in ['VT_Start', 'Time', 'timestamp']:
                if ts_col in df_ev.columns:
                    df_ev[ts_col] = pd.to_datetime(df_ev[ts_col], format='mixed')
            key = f"{asset_name}_{label}"
            events_data[key] = df_ev
            print(f"   ✓ Loaded  {match[0]:50s}  →  shape {df_ev.shape}")
        else:
            print(f"   ✗ No {suffix} file found")

print(f"\n{'='*60}")
print(f"Total event tables loaded: {len(events_data)}")


── Asset 1A (8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3) ──
   Files in events folder : ['8b999a4a-2a1e-4d6c-b76d-7a498fb5c4a3.parquet', '2021011814_2026011001_EAE.parquet', '2021011814_2026011001_EA.parquet', '2021011814_2026011001_EALE.parquet', '2021011814_2026011001_EAL.parquet', '2021011814_2026011001_E.parquet']
   ✓ Loaded  2021011814_2026011001_E.parquet                     →  shape (2507555, 30)
   ✓ Loaded  2021011814_2026011001_EA.parquet                    →  shape (54844, 24)
   ✓ Loaded  2021011814_2026011001_EAL.parquet                   →  shape (201117, 29)

── Asset 1O (df0dd88a-ed6d-412b-9d36-6c35829939cd) ──
   Files in events folder : ['2021011814_2026011001_EAE.parquet', 'df0dd88a-ed6d-412b-9d36-6c35829939cd.parquet', '2021011814_2026011001_EA.parquet', '2021011814_2026011001_EALE.parquet', '2021011814_2026011001_EAL.parquet', '2021011814_2026011001_E.parquet']
   ✓ Loaded  2021011814_2026011001_E.parquet                     →  shape (197161, 30)
   ✓ Loaded  202101181

### 7a. Events Data — Preview

In [40]:
for key, df_ev in events_data.items():
    print(f"\n── {key} ──")
    print(f"   Shape   : {df_ev.shape}")
    print(f"   Columns : {df_ev.columns.tolist()}")
    display(df_ev.head(3))


── 1A_ChangeEvents ──
   Shape   : (2507555, 30)
   Columns : ['Action', 'Actor', 'AreaName', 'AlarmLimit', 'Block', 'Category', 'ConditionName', 'Description', 'EventID', 'Flags', 'LocalTime', 'LocationFullName', 'LocationTagName', 'PrevValue', 'Priority', 'ReceivedDelay', 'ServerName', 'ShelvedReason', 'Source', 'SourceParameter', 'Station', 'Time', 'TransactionID', 'Units', 'Value', 'VT_Start', 'H', 'TagID', 'AlarmStatus', 'S']


,Action,Actor,AreaName,AlarmLimit,Block,Category,ConditionName,Description,EventID,Flags,...,Station,Time,TransactionID,Units,Value,VT_Start,H,TagID,AlarmStatus,S
0,None,None,1A,NaN,None,14,None,PROCESS EVNT RCVRY 1A ...,31214430,NaN,...,None,1974-03-17 20:27:01.115608,31214430,None,None,2021-10-11 10:31:51.560800,2021_10_11_10,$CONSOLE01,None,S1-PROD-ESVT01
1,None,None,1A,NaN,None,14,None,PROCESS EVNT RCVRY 1A ...,31214424,NaN,...,None,1974-03-17 20:27:01.116026,31214424,None,None,2021-10-11 10:31:51.602600,2021_10_11_10,$CONSOLE01,None,S1-PROD-ESVT01
2,None,None,1A,NaN,None,14,None,PROCESS EVNT RCVRY 1A ...,31214423,NaN,...,None,1974-03-17 20:27:01.116028,31214423,None,None,2021-10-11 10:31:51.602800,2021_10_11_10,$CONSOLE01,None,S1-PROD-ESVT01



── 1A_AlarmEvents ──
   Shape   : (54844, 24)
   Columns : ['ServerName', 'EventID', 'IntervalIdentifier', 'Priority', 'Parameter', 'FromValue', 'ToValue', 'Value', 'Description', 'ACTION', 'Unit', 'UnitOfMeasure', 'TagName', 'Area', 'Limit', 'LocationFullName', 'LocationTagName', 'AlarmStatus', 'VT_Start', 'TagID', 'EventTypeID', 'VT_End', 'H', 'S']


,ServerName,EventID,IntervalIdentifier,Priority,Parameter,FromValue,ToValue,Value,Description,ACTION,...,Limit,LocationFullName,LocationTagName,AlarmStatus,VT_Start,TagID,EventTypeID,VT_End,H,S
0,S1-PROD-ESVT01,61059050,CHANGE,16,None,None,INACTIVE,INACTIVE,PTEXECST,None,...,NaN,/Assets/TRAIN_1/1A,1A,None,2023-05-04 13:21:48.203000,$DET0108,4,None,2023_05_04_13,S1-PROD-ESVT01
1,S1-PROD-ESVT01,61059051,None,16,None,None,None,None,PTEXECST ACTIVE INACTIVE,None,...,NaN,/Assets/TRAIN_1/1A,1A,None,2023-05-04 13:21:48.203000,$DET0108,4,None,2023_05_04_13,S1-PROD-ESVT01
2,S1-PROD-ESVT01,61059053,CHANGE,16,None,None,INACTIVE,INACTIVE,PTEXECST,None,...,NaN,/Assets/TRAIN_1/1A,1A,None,2023-05-04 13:21:55.256300,$DET0108,4,None,2023_05_04_13,S1-PROD-ESVT01



── 1A_AlarmEventsLinked ──
   Shape   : (201117, 29)
   Columns : ['ServerName', 'EventID', 'IntervalIdentifier', 'Priority', 'Parameter', 'FromValue', 'ToValue', 'Value', 'Description', 'ACTION', 'Unit', 'UnitOfMeasure', 'TagName', 'Area', 'Limit', 'LocationFullName', 'LocationTagName', 'AlarmStatus', 'VT_Start', 'TagID', 'EventTypeID', 'VT_End', 'LastEvent', 'NextEvent', 'rowid', 'Seconds', 'PrevSeconds', 'H', 'S']


...

,Action,Actor,AreaName,AlarmLimit,Block,Category,ConditionName,Description,EventID,Flags,...,Station,Time,TransactionID,Units,Value,VT_Start,H,TagID,AlarmStatus,S
0,None,None,1O,NaN,None,14,None,PROCESS EVNT RCVRY 1O ...,31214647,NaN,...,None,1974-03-17 20:27:02.296024,31214647,None,None,2021-10-11 10:33:49.602400,2021_10_11_10,$CONSOLE01,None,S1-PROD-ESVT01
1,None,None,1O,NaN,None,14,None,PROCESS EVNT RCVRY 1O ...,31214648,NaN,...,None,1974-03-17 20:27:02.296024,31214648,None,None,2021-10-11 10:33:49.602400,2021_10_11_10,$CONSOLE01,None,S1-PROD-ESVT01
2,None,None,1O,NaN,None,14,None,PROCESS EVNT RCVRY 1O ...,31214649,NaN,...,None,1974-03-17 20:27:02.300150,31214649,None,None,2021-10-11 10:33:50.015000,2021_10_11_10,$CONSOLE01,None,S1-PROD-ESVT01



── 1O_AlarmEvents ──
   Shape   : (49571, 24)
   Columns : ['ServerName', 'EventID', 'IntervalIdentifier', 'Priority', 'Parameter', 'FromValue', 'ToValue', 'Value', 'Description', 'ACTION', 'Unit', 'UnitOfMeasure', 'TagName', 'Area', 'Limit', 'LocationFullName', 'LocationTagName', 'AlarmStatus', 'VT_Start', 'TagID', 'EventTypeID', 'VT_End', 'H', 'S']


,ServerName,EventID,IntervalIdentifier,Priority,Parameter,FromValue,ToValue,Value,Description,ACTION,...,Limit,LocationFullName,LocationTagName,AlarmStatus,VT_Start,TagID,EventTypeID,VT_End,H,S
0,S1-PROD-ESVT01,46282215,MESSAGE,16,None,None,None,None,20May22 17:33:04 * Completed AM Schedule Data ...,ACK,...,NaN,/Assets/TRAIN_1/1O,1O,None,2022-05-20 19:04:29.147800,$Prsts30,4,None,2022_05_20_19,S1-PROD-ESVT01
1,S1-PROD-ESVT01,31214650,None,16,None,None,None,None,UT ALM RECOV ...,None,...,NaN,/Assets/TRAIN_1/1O,1O,None,2021-10-11 10:33:50.013900,$UNITOPS,4,None,2021_10_11_10,S1-PROD-ESVT01
2,S1-PROD-ESVT01,31214651,None,16,None,None,None,None,UT RECOV CMP ...,None,...,NaN,/Assets/TRAIN_1/1O,1O,None,2021-10-11 10:33:50.013900,$UNITOPS,4,None,2021_10_11_10,S1-PROD-ESVT01



── 1O_AlarmEventsLinked ──
   Shape   : (11699, 29)
   Columns : ['ServerName', 'EventID', 'IntervalIdentifier', 'Priority', 'Parameter', 'FromValue', 'ToValue', 'Value', 'Description', 'ACTION', 'Unit', 'UnitOfMeasure', 'TagName', 'Area', 'Limit', 'LocationFullName', 'LocationTagName', 'AlarmStatus', 'VT_Start', 'TagID', 'EventTypeID', 'VT_End', 'LastEvent', 'NextEvent', 'rowid', 'Seconds', 'PrevSeconds', 'H', 'S']


,ServerName,EventID,IntervalIdentifier,Priority,Parameter,FromValue,ToValue,Value,Description,ACTION,...,TagID,EventTypeID,VT_End,LastEvent,NextEvent,rowid,Seconds,PrevSeconds,H,S
0,S1-PROD-ESVT01,31973122,UNCEVT,16,None,None,CLOSE,CLOSE,3G116 GLYCOL BAK FLOW,None,...,03ESDV_1669,1,None,0,2,1,7,6,2021_11_25_09,S1-PROD-ESVT01
1,S1-PROD-ESVT01,31973133,UNCEVT,16,None,None,CLOSE,CLOSE,3G116 GLYCOL BAK FLOW,OK,...,03ESDV_1669,2,None,1,1,2,555436,7,2021_11_25_09,S1-PROD-ESVT01
2,S1-PROD-ESVT01,32512391,UNCEVT,16,None,None,MOVING,MOVING,3G116 GLYCOL BAK FLOW,None,...,03ESDV_1669,1,None,2,2,3,13,555436,2021_12_01_19,S1-PROD-ESVT01


## 8. Summary

In [41]:
print(f"{'='*60}")
print(f"  SUMMARY for target tag: {TARGET_TAG}")
print(f"{'='*60}")
print(f"  Related tags in sheet         : {len(df_tags)}")
print(f"  Unique assets referenced      : {len(df_asset_map)}")
print(f"  Assets found in Historian/    : {df_availability['In Historian/'].sum()}")
print(f"  Assets found in events/       : {df_availability['In events/'].sum()}")
print(f"  Historian series loaded       : {len(historian_data)}")
print(f"  Event tables loaded           : {len(events_data)}")
print(f"{'='*60}")
print()
print("Historian series:")
for k, v in historian_data.items():
    print(f"  {k:35s}  {v.shape[0]:>8,} rows")
print()
print("Event tables:")
for k, v in events_data.items():
    print(f"  {k:45s}  {v.shape[0]:>8,} rows")

  SUMMARY for target tag: 03LIC_1619
  Related tags in sheet         : 12
  Unique assets referenced      : 2
  Assets found in Historian/    : 2
  Assets found in events/       : 2
  Historian series loaded       : 12
  Event tables loaded           : 6

Historian series:
  02FI_1000.PV                         2,130,628 rows
  03LIC_1619.PV                        2,134,309 rows
  03LIC_1608.PV                        2,134,309 rows
  03PIC_1620.PV                        2,134,280 rows
  03FIC_1668.PV                        2,112,127 rows
  03TIC_1671.PV                        2,134,292 rows
  03LIC_1603.OP                        2,134,187 rows
  03LIC_1618.OP                        2,134,187 rows
  03GM_0116_I.PV                       2,134,349 rows
  03GM_0116A_I.PV                      2,134,349 rows
  03PDI_1611.PV                        2,134,299 rows
  03TIC_1635.PV                        2,134,126 rows

Event tables:
  1A_ChangeEvents                                2,507,555 rows

## 9. Combine All Events and Save

Concatenate all event tables (`_E`, `_EA`, `_EAL`) for all assets into a single DataFrame,
add a `source_table` column to track the origin, sort by `VT_Start`, and save to
`DATA/combined_events/<TARGET_TAG>_combined_events.parquet`.

In [42]:
import os

# ── Output directory ────────────────────────────────────────────────────────────
OUTPUT_DIR = os.path.join(DATA_DIR, 'combined_events')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory : {OUTPUT_DIR}")
print()

# ── Concatenate all event DataFrames ────────────────────────────────────────────
print(f"Concatenating {len(events_data)} event tables ...\n")

parts = []
for table_key, df_part in events_data.items():
    df_part = df_part.copy()
    df_part['source_table'] = table_key          # track origin: e.g. '1E_AlarmEvents'
    parts.append(df_part)
    print(f"  + {table_key:45s}  {len(df_part):>9,} rows  |  "
          f"VT_Start range: {df_part['VT_Start'].min()}  →  {df_part['VT_Start'].max()}")

df_combined = pd.concat(parts, ignore_index=True, sort=False)

print(f"\nRaw combined shape : {df_combined.shape}")
print(f"Columns            : {df_combined.columns.tolist()}")

# ── Apply timezone offset ─────────────────────────────────────────────────────────
# Historian events are stored in UTC+5:30 (IST); plant PV data is in UTC+4 (UAE local).
# Subtract 1.5 hours to align event timestamps with the PV time series.
TIME_OFFSET = pd.Timedelta(hours=1.5)
print(f"\nApplying timezone offset: VT_Start -= {TIME_OFFSET} (UTC+5:30 → UAE local UTC+4)")
print(f"  VT_Start before: {df_combined['VT_Start'].min()}  →  {df_combined['VT_Start'].max()}")
df_combined['VT_Start'] = df_combined['VT_Start'] - TIME_OFFSET
print(f"  VT_Start after : {df_combined['VT_Start'].min()}  →  {df_combined['VT_Start'].max()}")

# ── Sort by VT_Start ─────────────────────────────────────────────────────────────
print(f"\nSorting by VT_Start ...")
df_combined.sort_values('VT_Start', inplace=True, ignore_index=True)

print(f"Sorted combined shape : {df_combined.shape}")
print(f"VT_Start range        : {df_combined['VT_Start'].min()}  →  {df_combined['VT_Start'].max()}")
print(f"Null VT_Start rows    : {df_combined['VT_Start'].isna().sum():,}")

# ── Save ─────────────────────────────────────────────────────────────────────────
output_path = os.path.join(OUTPUT_DIR, f"{TARGET_TAG}_combined_events.parquet")
df_combined.to_parquet(output_path, index=False)
file_size_mb = os.path.getsize(output_path) / (1024 ** 2)

print(f"\n{'='*60}")
print(f"  Saved to : {output_path}")
print(f"  Rows     : {len(df_combined):,}")
print(f"  Columns  : {len(df_combined.columns)}")
print(f"  File size: {file_size_mb:.1f} MB")
print(f"{'='*60}")
print(f"\nsource_table breakdown:")
print(df_combined['source_table'].value_counts().to_string())

Output directory : /home/h604827/ControlActions/DATA/combined_events

Concatenating 6 event tables ...

  + 1A_ChangeEvents                                2,507,555 rows  |  VT_Start range: 2021-02-01 17:19:41.804300  →  2025-06-28 00:22:59.137300
  + 1A_AlarmEvents                                    54,844 rows  |  VT_Start range: 2021-10-02 14:30:37.668800  →  2025-06-27 18:53:34.052000
  + 1A_AlarmEventsLinked                             201,117 rows  |  VT_Start range: 2021-10-09 09:58:38.952700  →  2025-06-03 10:22:26.768300
  + 1O_ChangeEvents                                  197,161 rows  |  VT_Start range: 2021-10-02 01:35:35.153800  →  2025-06-28 05:27:42.753600
  + 1O_AlarmEvents                                    49,571 rows  |  VT_Start range: 2021-10-03 00:52:07.054200  →  2025-06-28 05:19:34.367700
  + 1O_AlarmEventsLinked                              11,699 rows  |  VT_Start range: 2021-10-12 22:38:10.353800  →  2025-06-28 05:19:24.255900

Raw combined shape : (3021947, 